# CountZES — Run `+ MD-FCE` riêng lẻ trên `MBM` (Bộ nền GitHub gốc)

Notebook này chạy biến thể **`+ MD-FCE` (`--md_fce --md_fce_gamma 0.0`)** độc lập trên tập **`MBM` (`44/44` ảnh, `CounTR + FSC147.pth`)** trên **bộ nền mặc định GitHub gốc (`2026-11-26`)**, lưu bền vững tại Google Drive `/content/drive/MyDrive/CountZES/`.

**Cấu hình chạy chính:**
- Script: `countzes_mbm.py --fsoc countr --countr_ckpt ./CounTR/output_allnew_dir/FSC147.pth`
- Nền cố định: `--box_threshold 0.20 --text_threshold 0.15 --sses_start_percentile 80 --github_sses_inversion --single_instance_mode paper --clip_sim_mode raw`
- Đề xuất bật riêng lẻ: `--md_fce --md_fce_gamma 0.0`
- Output lưu tại: `MyDrive/CountZES/outputs/mbm_gh_raw_1sim_mdfce0/`

**Trước khi chạy:** Chọn **Runtime → Change runtime type → T4 GPU** (hoặc trong Antigravity IDE chọn **Select Kernel → Colab → Auto Connect**).

In [ ]:
# [CELL-1] Kiểm tra GPU & thiết lập biến môi trường PyTorch
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

!nvidia-smi
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# [CELL-2] Clone hoặc cập nhật code mới nhất từ GitHub (idempotent)
import getpass, subprocess

REPO_OWNER_PATH = "github.com/duylw/cv-advanced.git"
REPO_URL = f"https://{REPO_OWNER_PATH}"
REPO_DIR = "/content/cv-advanced"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    env_no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], env=env_no_prompt)
    if res.returncode != 0:
        print("Repo đang ở chế độ Private -> nhập GitHub PAT (token không được lưu vào file):")
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
        subprocess.run(["git", "clone", f"https://{token}@{REPO_OWNER_PATH}", REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
else:
    env_no_prompt = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], env=env_no_prompt)
    if res.returncode != 0:
        token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub PAT: ")
        subprocess.run(["git", "-C", REPO_DIR, "pull", f"https://{token}@{REPO_OWNER_PATH}", "main"], check=True)

%cd {REPO_DIR}
!git log -n 2 --oneline

In [ ]:
# [CELL-3] Cài thư viện còn thiếu & kiểm tra cú pháp các script Python
!python -c "import addict, yapf, pycocotools, termcolor, yaml, gdown, supervision, timm, einops, transformers, psutil" 2>/dev/null || pip install -q opencv-python tqdm matplotlib timm einops supervision sentencepiece accelerate scipy pandas scikit-image addict yapf pycocotools termcolor pyyaml gdown psutil "transformers>=4.31.0"
!python -c "import segment_anything" 2>/dev/null || pip install -q git+https://github.com/facebookresearch/segment-anything.git

!python -m py_compile resource_monitor.py countzes_core.py countzes_fsc147.py countzes_carpk.py countzes_persense_D.py countzes_mbm.py countzes_vgg.py

In [ ]:
# [CELL-4] Mount Google Drive & symlink data/outputs về MyDrive/CountZES
import shutil
from google.colab import drive

drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CountZES"
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data")
DRIVE_WEIGHTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "weights")
DRIVE_OUTPUTS_DIR = os.path.join(DRIVE_PROJECT_DIR, "outputs")

for d in [DRIVE_PROJECT_DIR, DRIVE_DATA_DIR, DRIVE_WEIGHTS_DIR, DRIVE_OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

for sub in ["vgg", "mbm", "persenseD"]:
    os.makedirs(os.path.join(DRIVE_DATA_DIR, sub), exist_ok=True)
    local_sub = os.path.join("data", sub)
    if os.path.islink(local_sub):
        os.unlink(local_sub)
    elif os.path.exists(local_sub):
        shutil.rmtree(local_sub)
    os.symlink(os.path.join(DRIVE_DATA_DIR, sub), local_sub)

if os.path.islink("outputs"):
    os.unlink("outputs")
elif os.path.exists("outputs"):
    shutil.rmtree("outputs")
os.symlink(DRIVE_OUTPUTS_DIR, "outputs")

print("Mounted project folder:", DRIVE_PROJECT_DIR)
print(" - Data   :", DRIVE_DATA_DIR)
print(" - Weights:", DRIVE_WEIGHTS_DIR)
print(" - Outputs:", DRIVE_OUTPUTS_DIR)

In [ ]:
# [CELL-5] Kiểm tra / tải Weights vào Drive (idempotent) và symlink vào repo
os.makedirs("DSALVANet/checkpoints", exist_ok=True)
os.makedirs("GroundingDINO/weights", exist_ok=True)
os.makedirs("CounTR/output_allnew_dir", exist_ok=True)

def ensure_weight(weight_name, local_path, download_cmd=None):
    drive_path = os.path.join(DRIVE_WEIGHTS_DIR, weight_name)
    if not (os.path.exists(drive_path) and os.path.getsize(drive_path) > 0):
        if download_cmd:
            print(f"[Download -> Drive] {drive_path}")
            get_ipython().system(download_cmd.format(dst=drive_path))
        else:
            print(f"[Optional] Chưa có {weight_name} trong {DRIVE_WEIGHTS_DIR}")
            return
    else:
        print(f"[Drive OK] {drive_path}")
    if not (os.path.exists(drive_path) and os.path.getsize(drive_path) > 0):
        print(f"[WARNING] Chưa tải được {weight_name}. Hãy đặt file vào: {drive_path}")
        return

    if os.path.islink(local_path) or os.path.exists(local_path):
        os.remove(local_path)
    os.symlink(drive_path, local_path)
    print(f"  -> Linked {local_path}")

# 1. SAM ViT-H (~2.56 GB)
ensure_weight(
    "sam_vit_h_4b8939.pth",
    "data/sam_vit_h_4b8939.pth",
    "wget -q --show-progress -O '{dst}' https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
)

# 2. GroundingDINO SwinT-OGC
ensure_weight(
    "groundingdino_swint_ogc.pth",
    "GroundingDINO/weights/groundingdino_swint_ogc.pth",
    "gdown 13rV6dzRwWCVZYWpnmiaVwRDIDC28d82g -O '{dst}'"
)

# 3. CounTR pretrained FSC-147 (dùng cho track MBM cross-domain)
ensure_weight(
    "FSC147.pth",
    "CounTR/output_allnew_dir/FSC147.pth",
    "gdown 1CzYyiYqLshMdqJ9ZPFJyIzXBa7uFUIYZ -O '{dst}'"
)

In [ ]:
# [CELL-6] Đảm bảo dataset MBM sẵn sàng trong MyDrive/CountZES/data/mbm/Images/
import glob, subprocess, tarfile, zipfile
from collections import Counter

def _extract_any(archive_path, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    if zipfile.is_zipfile(archive_path):
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(dst_dir)
        return "zip"
    if tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path, "r:*") as tf:
            tf.extractall(dst_dir)
        return "tar"
    with open(archive_path, "rb") as f:
        head = f.read(16)
    for cmd in (["7z", "x", "-y", f"-o{dst_dir}", archive_path], ["unrar", "x", "-o+", archive_path, dst_dir]):
        if shutil.which(cmd[0]):
            res = subprocess.run(cmd, capture_output=True, text=True)
            if res.returncode == 0:
                return cmd[0]
    raise RuntimeError(f"Không giải nén được {archive_path} (magic={head!r})")

def ensure_mbm_dataset(file_id="1i54Hcw2GbCVp0Kz7jei2Lr7Cmi2DZkol"):
    target_images_dir = os.path.join(DRIVE_DATA_DIR, "mbm", "Images")
    if os.path.exists(target_images_dir) and any(os.scandir(target_images_dir)):
        print(f"[Drive OK] Đã có sẵn MBM dataset: {target_images_dir}")
        return
    zip_path = "/content/mbm.zip"
    tmp_dir = "/content/_tmp_mbm"
    shutil.rmtree(tmp_dir, ignore_errors=True)
    if not (os.path.exists(zip_path) and os.path.getsize(zip_path) > 0):
        get_ipython().system(f"gdown {file_id} -O {zip_path}")
    if not (os.path.exists(zip_path) and os.path.getsize(zip_path) > 0):
        raise RuntimeError(f"Không tải được mbm.zip bằng gdown (file_id={file_id}).")

    fmt = _extract_any(zip_path, tmp_dir)
    print(f"Đã giải nén {zip_path} (định dạng: {fmt})")
    all_files = [p for p in glob.glob(os.path.join(tmp_dir, "**", "*"), recursive=True) if os.path.isfile(p) and "__MACOSX" not in p and not os.path.basename(p).startswith("._")]
    if len(all_files) <= 2 and all_files and all_files[0].lower().endswith((".zip", ".tar", ".gz", ".tgz", ".rar", ".7z")):
        nested_dir = os.path.join(tmp_dir, "_nested")
        _extract_any(all_files[0], nested_dir)
        all_files = [p for p in glob.glob(os.path.join(nested_dir, "**", "*"), recursive=True) if os.path.isfile(p) and "__MACOSX" not in p and not os.path.basename(p).startswith("._")]

    dots_files = [p for p in all_files if p.lower().endswith("dots.png")]
    mask_files = [p for p in all_files if p.endswith(("A.png", "a.png"))]
    os.makedirs(target_images_dir, exist_ok=True)
    dst_cdir = os.path.join(target_images_dir, "blood cells")
    os.makedirs(dst_cdir, exist_ok=True)

    if mask_files:
        for cdir in sorted({os.path.dirname(p) for p in mask_files}):
            for f in os.listdir(cdir):
                src_f = os.path.join(cdir, f)
                if os.path.isfile(src_f) and not f.startswith("._"):
                    out_name = f[:-5] + "A.png" if f.endswith("a.png") else f
                    shutil.copy2(src_f, os.path.join(dst_cdir, out_name))
    elif dots_files:
        copied = 0
        for dp in sorted(dots_files):
            prefix = os.path.basename(dp)[:-len("dots.png")]
            cell_cand = os.path.join(os.path.dirname(dp), f"{prefix}cell.png")
            if os.path.exists(cell_cand):
                shutil.copy2(cell_cand, os.path.join(dst_cdir, f"{prefix}.png"))
                shutil.copy2(dp, os.path.join(dst_cdir, f"{prefix}A.png"))
                copied += 1
        assert copied > 0, f"Thấy {len(dots_files)} file *dots.png nhưng không khớp cặp *cell.png"
    else:
        raise AssertionError(f"Không tìm thấy ảnh hợp lệ trong mbm.zip (mẫu: {[os.path.relpath(p, tmp_dir) for p in sorted(all_files)[:10]]})")

    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"[Dataset Ready] {target_images_dir}")

ensure_mbm_dataset()
!ls -la data/mbm/Images/"blood cells" | head -n 10

In [ ]:
# [CELL-7] Chạy MBM (44/44) với + MD-FCE riêng lẻ trên bộ nền GitHub gốc
COUNTR_CKPT = "./CounTR/output_allnew_dir/FSC147.pth"
assert os.path.exists(COUNTR_CKPT), f"Thiếu checkpoint CounTR: {COUNTR_CKPT}"

OUTDIR = "mbm_gh_raw_1sim_mdfce0"   # Lưu tại MyDrive/CountZES/outputs/mbm_gh_raw_1sim_mdfce0/
MD_FCE_GAMMA = 0.0                  # 0.0 = lọc thuần mask SAM (chuẩn Pha 3); có thể thử 1.0 nếu muốn nhấn mật độ

CMD = (
    f"python countzes_mbm.py "
    f"--data ./data "
    f"--fsoc countr "
    f"--countr_ckpt {COUNTR_CKPT} "
    f"--box_threshold 0.20 "
    f"--text_threshold 0.15 "
    f"--sses_start_percentile 80 "
    f"--github_sses_inversion "
    f"--single_instance_mode paper "
    f"--clip_sim_mode raw "
    f"--md_fce "
    f"--md_fce_gamma {MD_FCE_GAMMA} "
    f"--stage_samples -1 "
    f"--visualize "
    f"--outdir {OUTDIR}"
)
print("Running:", CMD)
!{CMD}

In [ ]:
# [CELL-8] Kiểm tra summary.json, resource log, predictions.csv và ảnh stage_samples trên Drive
import glob, json
import pandas as pd
from IPython.display import Image, display

run_dir = os.path.join(DRIVE_OUTPUTS_DIR, OUTDIR)
summary_path = os.path.join(run_dir, "summary.json")
res_log_path = os.path.join(run_dir, "logs", "resource_usage.log")
csv_path = os.path.join(run_dir, "predictions.csv")

if os.path.exists(summary_path):
    print("=== SUMMARY JSON ===")
    print(open(summary_path, encoding="utf-8").read())
if os.path.exists(res_log_path):
    print("=== RESOURCE LOG ===")
    print(open(res_log_path, encoding="utf-8").read())
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"=== TOP 5 ẢNH SAI SỐ LỚN NHẤT ({len(df)} ảnh) ===")
    cols = [c for c in ["filename", "gt_count", "pred_count", "abs_error", "pseudo_gt_kde", "fce_centroid_sep", "duration_sec", "status"] if c in df.columns]
    display(df.sort_values("abs_error", ascending=False)[cols].head(5))

stage_imgs = sorted(glob.glob(f"{run_dir}/stage_samples/*.png"))[:6]
print(f"Tìm thấy {len(stage_imgs)} ảnh phân tích giai đoạn (2x4) trong {run_dir}/stage_samples/:")
for p in stage_imgs:
    print(p)
    display(Image(filename=p))